# Variational Autoencoder (VAE): From-Scratch Walkthrough

This notebook implements and trains a **convolutional VAE** on MNIST, with all probabilistic components built from scratch:

| Component | What it does |
|---|---|
| Gaussian log-likelihood | Scores how well reconstructions match inputs |
| KL divergence (exact) | Measures encoder drift from the standard-normal prior |
| Reparameterization trick | Enables gradient flow through the stochastic sampling step |
| ELBO loss | Balances reconstruction quality against latent regularization |

The neural network layers (Conv2d, Linear, etc.) use standard PyTorch modules; the pedagogical focus is on the **VAE-specific math**, not reimplementing convolutions.

**Task:** Generative modeling of handwritten digits.
**Dataset:** MNIST (28 × 28 grayscale, 60 k train / 10 k test), auto-downloaded via torchvision.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from vae_from_scratch import VAE, log_prob, kl_q_p_exact, rsample

## Architecture Overview

The VAE consists of two networks and a stochastic bridge between them:

```
Input image (1, 28, 28)
        │
   ┌────▼────┐
   │ Encoder │  Conv2d → ReLU → Conv2d → ReLU → Flatten → Linear
   └────┬────┘
        │
        ▼
   [μ, log σ]  ←  K+1 values defining q(z|x) = N(μ, σ²I)
        │
        ▼
   z = μ + σ · ε,   ε ~ N(0, I)   ←  Reparameterization trick
        │
   ┌────▼────┐
   │ Decoder │  Linear → Unflatten → ReLU → ConvTranspose2d → ReLU → ConvTranspose2d
   └────┬────┘
        │
        ▼
   Reconstructed image (1, 28, 28)
```

**Key design choice:** The encoder outputs a *single shared* log σ for all latent dimensions (isotropic Gaussian), rather than per-dimension variances. This simplifies the KL computation and works well for low-dimensional latent spaces (K = 2 here).

## The Reparameterization Trick

**The problem:** VAEs need to *sample* $z$ from $q(z \mid x)$ during training, but sampling is not differentiable; gradients can't flow through a random draw.

**The solution:** Instead of sampling $z \sim \mathcal{N}(\mu, \sigma^2 I)$ directly, we:

1. Sample $\varepsilon \sim \mathcal{N}(0, I)$, a fixed distribution with no learnable parameters.
2. Compute $z = \mu + \sigma \cdot \varepsilon$, a deterministic function of $\varepsilon$.

Now gradients flow through $\mu$ and $\sigma$ back to the encoder, because the randomness is isolated in $\varepsilon$ (which doesn't depend on any parameters).

$$z = \mu + \sigma \cdot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I)$$

This is implemented in `rsample()` in our module.

## ELBO Loss

The VAE is trained by maximizing the **Evidence Lower Bound (ELBO)**:

$$\text{ELBO} = \underbrace{\mathbb{E}_{q(z \mid x)}[\log p(x \mid z)]}_{\text{reconstruction term}} \;-\; \underbrace{KL\!\bigl(q(z \mid x)\;\|\;p(z)\bigr)}_{\text{regularization term}}$$

In practice we *minimize* the **negative** ELBO:

$$\mathcal{L} = -\mathbb{E}_{q(z \mid x)}[\log p(x \mid z)] \;+\; KL\!\bigl(q(z \mid x)\;\|\;p(z)\bigr)$$

**Reconstruction term:** How well does the decoder reconstruct $x$ from $z$? Measured as the Gaussian log-likelihood of $x$ given the decoder output $\mu_x$.

**KL term:** How close is the encoder distribution $q(z \mid x)$ to the standard-normal prior $p(z) = \mathcal{N}(0, I)$? Uses the exact closed-form for isotropic Gaussians:

$$KL(\mathcal{N}_q \| \mathcal{N}_p) = \frac{1}{2}\!\left[K\frac{\sigma_q^2}{\sigma_p^2} + \frac{\|\mu_q - \mu_p\|^2}{\sigma_p^2} - K + 2K\log\frac{\sigma_p}{\sigma_q}\right]$$

The KL term acts as a regularizer: without it, the encoder could map each input to a point (zero variance), collapsing to a standard autoencoder with no useful latent structure.

In [ ]:
# MNIST dataset; auto-downloads on first run (~12 MB)
transform = transforms.ToTensor()  # PIL [0, 255] -> Tensor [0.0, 1.0]

train_set = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
test_set  = datasets.MNIST(root="./data", train=False, transform=transform, download=True)

train_loader = DataLoader(train_set, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)

print(f"Train: {len(train_set)} images  |  Test: {len(test_set)} images")
print(f"Image shape: {train_set[0][0].shape}")  # (1, 28, 28)

In [ ]:
# Preview a few training images
fig, axes = plt.subplots(1, 8, figsize=(12, 1.5))
for i, ax in enumerate(axes):
    img, label = train_set[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(str(label), fontsize=10)
    ax.axis("off")
fig.suptitle("Sample training images", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
LATENT_DIM = 2  # Low dimensionality lets us visualize the latent space in 2D

model = VAE(latent_dim=LATENT_DIM, num_filters=32)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"VAE with K={LATENT_DIM}: {n_params:,} parameters")
print(f"Learned decoder log_sigma_x initialized to: {model.log_sig_x.item():.4f}")

## Training

We minimize the negative ELBO. The loop below computes reconstruction and KL terms separately so we can track them independently; useful for diagnosing training dynamics (e.g., *KL collapse*, where the KL term drops to zero and the model ignores the latent space).

In [ ]:
def train_vae(model, train_loader, epochs=10, n_samples=1, lr=1e-3):
    '''Train the VAE and return per-epoch loss history.'''
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()

    history = {"total": [], "recon": [], "kl": []}

    for epoch in range(epochs):
        epoch_total, epoch_recon, epoch_kl = 0.0, 0.0, 0.0
        n_batches = 0

        for images, _ in train_loader:
            # --- Forward pass (decomposed for tracking) ---
            phi = model.encode(images)                          # (B, K+1)
            z = rsample(phi, n_samples)                         # (B, N, K)
            mu_x = model.decode(z)                              # (B, N, 1, 28, 28)

            B, C, H, W = images.shape
            x_flat = images.view(B, 1, -1).expand(-1, n_samples, -1)
            mu_x_flat = mu_x.view(B, n_samples, -1)
            log_sig = model.log_sig_x.view(1, 1).expand(B, n_samples)

            # Reconstruction: E_q[ log p(x|z) ]
            recon = log_prob(x_flat, mu_x_flat, log_sig).mean()
            # KL: KL( q(z|x) || p(z) )
            kl = kl_q_p_exact(phi, torch.zeros_like(phi)).mean()

            loss = -recon + kl  # negative ELBO

            # --- Backward pass ---
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_total += loss.item()
            epoch_recon += (-recon).item()
            epoch_kl += kl.item()
            n_batches += 1

        # Per-epoch averages
        history["total"].append(epoch_total / n_batches)
        history["recon"].append(epoch_recon / n_batches)
        history["kl"].append(epoch_kl / n_batches)

        print(
            f"Epoch {epoch + 1:2d}/{epochs}  |  "
            f"Loss: {history['total'][-1]:8.2f}  |  "
            f"Recon: {history['recon'][-1]:8.2f}  |  "
            f"KL: {history['kl'][-1]:6.2f}  |  "
            f"σ_x: {model.log_sig_x.exp().item():.4f}"
        )

    model.eval()
    return history


# Train; expect ~3-4 min on CPU with n_samples=1
history = train_vae(model, train_loader, epochs=10, n_samples=1, lr=1e-3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(history["total"], "b-o", markersize=4)
axes[0].set_title("Total Loss (negative ELBO)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(history["recon"], "r-o", markersize=4)
axes[1].set_title("Reconstruction Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel(r"$-\mathbb{E}[\log\,p(x|z)]$")

axes[2].plot(history["kl"], "g-o", markersize=4)
axes[2].set_title("KL Divergence")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel(r"$KL(q \| p)$")

plt.suptitle("VAE Training Curves", fontsize=13)
plt.tight_layout()
plt.show()

## Generating New Samples

During training, $z$ is sampled from the encoder: $z \sim q(z \mid x)$. At inference time we have no input $x$, so we sample from the **prior**: $z \sim \mathcal{N}(0, I)$ and decode.

Note: Basic VAEs on MNIST typically produce **blurry** samples; this is a known limitation of the Gaussian decoder assumption and the ELBO objective, not a bug. GANs and diffusion models address this with different objectives and architectures.

In [ ]:
model.eval()
samples = model.generate(n_images=16)  # (16, 1, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(14, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].squeeze().detach().clamp(0, 1).numpy(), cmap="gray")
    ax.axis("off")
fig.suptitle("Generated samples  (z ~ N(0, I)  →  decoder)", fontsize=13)
plt.tight_layout()
plt.show()

## Latent Space Visualization

With $K = 2$ we can directly plot the 2D latent space. Each test image is encoded to its mean $\mu$ (ignoring variance), then plotted and colored by digit label.

A well-trained VAE should show **smooth clustering** with overlapping regions, reflecting the continuous, interpolatable structure the model learned.

In [ ]:
model.eval()
all_mu = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        phi = model.encode(images)      # (B, K+1)
        mu = phi[:, :-1]                # (B, K), encoder means only
        all_mu.append(mu)
        all_labels.append(labels)

all_mu = torch.cat(all_mu).numpy()          # (10000, 2)
all_labels = torch.cat(all_labels).numpy()  # (10000,)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    all_mu[:, 0], all_mu[:, 1],
    c=all_labels, cmap="tab10", alpha=0.5, s=4,
)
plt.colorbar(scatter, label="Digit", ticks=range(10))
plt.xlabel(r"$z_1$")
plt.ylabel(r"$z_2$")
plt.title("Latent space  (encoder means on test set, colored by digit)")
plt.tight_layout()
plt.show()

## Reconstructions

To verify the model has learned a meaningful encoding, we pass test images through the full encode → sample → decode pipeline and compare originals to reconstructions.

In [ ]:
model.eval()
test_batch, _ = next(iter(test_loader))
test_imgs = test_batch[:8]  # 8 images

with torch.no_grad():
    phi = model.encode(test_imgs)
    z = rsample(phi, 1)             # (8, 1, K)
    recon = model.decode(z)         # (8, 1, 1, 28, 28)
    recon = recon.squeeze(1)        # (8, 1, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(14, 3.5))
for i in range(8):
    axes[0, i].imshow(test_imgs[i].squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon[i].squeeze().clamp(0, 1).numpy(), cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=11, rotation=0, labelpad=55)
axes[1, 0].set_ylabel("Recon", fontsize=11, rotation=0, labelpad=55)
fig.suptitle("Input vs. Reconstruction", fontsize=13)
plt.tight_layout()
plt.show()

## Some things I Learned

- **The reparameterization trick is subtle but essential.** Without it, the sampling step blocks all gradient flow to the encoder. The trick ($z = \mu + \sigma\varepsilon$) is a simple algebraic rearrangement, but it's what makes VAE training via backpropagation possible.

- **Reconstruction vs. KL is a real tension.** Early in training the model often "ignores" the latent space (posterior collapse / KL vanishing) because the
  reconstruction loss dominates. The learnable decoder sigma ($\sigma_x$) helps balance these terms.

- **Blurry outputs are expected, not a bug.** The Gaussian decoder assumption means the model averages over modes rather than committing to sharp samples. This is a fundamental limitation of the basic VAE objective, addressed by richer decoder distributions, adversarial losses (VAE-GAN), or entirely different frameworks (diffusion models).

- **2D latent spaces are great for intuition, limited for quality.** $K = 2$ lets us visualize clustering and interpolation directly, but severely constrains capacity. Higher $K$ (e.g., 16 or 64) produces sharper generations at the cost of direct visualization.

- **The isotropic Gaussian assumption simplifies everything.** A single scalar $\sigma$ per sample (rather than per-dimension variances) makes the KL closed form trivial and reduces encoder output from $2K$ to $K + 1$ values. As mentioned in the lectures, in practice, diagonal-covariance encoders are more common.